# 00 · Setup Check — Verificação do Ambiente

🎯 **Objetivo:** Validar seu ambiente Python/PySpark e descobrir quais perfis Docker estão rodando.

Execute este notebook **antes de iniciar qualquer laboratório**. Ele é seguro para rodar a qualquer momento, com qualquer subconjunto de `make up-*` em execução (ou nenhum — o Caso A usa `local[*]` e não precisa de Docker).

💡 **Dica:** Use este notebook como um "check-up" sempre que trocar de máquina, recriar o ambiente virtual ou encontrar erros inesperados.

---
## ✅ O que este notebook verifica

1. **Versão do Python** — 3.12+ obrigatório
2. **Versão do PySpark** — 3.5.x obrigatório
3. **SparkSession local** — Caso A funcional
4. **Perfis Docker** — Casos B/C/D disponíveis?
5. **Dataset Bronze** — dados gerados?

Vamos começar!


In [ ]:
import sys

# Exibe a versão do Python em execução no momento
print(f"Python: {sys.version}")
# Garante que a versão atende ao requisito mínimo (3.12+) — aborta se for inferior
assert sys.version_info >= (3, 12), "This lab targets Python 3.12+"
print("✅ Python version OK")

📌 **O que acabamos de verificar:** A versão do Python (3.12+) é importante porque o PySpark 3.5.x tira proveito de otimizações internas que só existem a partir desta versão. Se você estiver com uma versão anterior, atualize antes de prosseguir.

💡 **Dica:** Para verificar manualmente no terminal: `python --version`


In [ ]:
import pyspark

# Exibe a versão do PySpark instalada no ambiente
print(f"PySpark: {pyspark.__version__}")
# Verifica se é a versão 3.5.x — única série testada para estes laboratórios
assert pyspark.__version__.startswith("3.5"), "Expected PySpark 3.5.x"
print("✅ PySpark version OK")

📌 **PySpark 3.5.x** é a versão LTS atual do ecossistema Spark. Ela traz suporte a Spark Connect (usado nos Casos B/D), melhorias no Optimizer Catalyst e integração nativa com Pandas via `toPandas()`. Manter a versão correta evita incompatibilidades com as APIs utilizadas nos laboratórios.


In [ ]:
# Caso A não precisa de Docker — local[*] vem dentro do pacote pip do pyspark.
import sys

# Adiciona o diretório scripts/ ao path para importar módulos auxiliares
sys.path.insert(0, "../scripts")
# get_local_session cria uma SparkSession configurada com local[*]
from lab_utils import get_local_session

spark = get_local_session("00-setup-check")
# Cria um DataFrame minúsculo de teste com números de 0 a 4
df = spark.range(5).toDF("n")
# Força uma ação (count) para validar que o Spark realmente processa os dados
assert df.count() == 5
print("✅ Local Spark session works (Case A ready)")
spark.stop()

✅ **Caso A pronto.** O modo `local[*]` cria uma SparkSession completa — com Driver e Executors rodando como **threads na sua máquina** — sem precisar de Docker. O `*` significa "use todos os núcleos disponíveis".

🧠 **Por quê?** Você pode executar todos os notebooks deste laboratório mesmo sem os contêineres em execução (embora os Casos B/C/D exijam Docker para explorar modos avançados de deploy).


## Quais perfis Docker estão ativos?

A célula abaixo verifica as portas que cada perfil expõe. Ela **não inicia nada** — apenas testa a conectividade com cada serviço já em execução.

💡 **Dica:** Se um perfil aparecer como ⬜ not running, execute `make up-<caso>` no terminal (ex: `make up-b` para o Caso B) e rode esta célula novamente.

In [ ]:
import socket


# Tenta abrir uma conexão TCP na porta; retorna True se conseguiu
def port_open(host: str, port: int, timeout: float = 1.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


# Dicionário mapeando nome legível do perfil -> (endereço, porta)
checks = {
    "Case B/D — Spark Connect (localhost:15002)": ("localhost", 15002),
    "Case B — Spark Master UI (localhost:8080)": ("localhost", 8080),
    "Case D — RustFS S3 API (localhost:9000)": ("localhost", 9000),
    "Case C — YARN ResourceManager (localhost:8088)": ("localhost", 8088),
    "Case C — HDFS HttpFS gateway (localhost:14000)": ("localhost", 14000),
}

# Varre todos os perfis e imprime o status de cada um
for label, (host, port) in checks.items():
    status = "✅ up" if port_open(host, port) else "⬜ not running"
    print(f"{status:12s} {label}")

📌 **Interpretação das portas:**
- **Spark Connect (15002):** necessário para os Casos B e D (modo cliente-servidor)
- **Spark Master UI (8080):** interface web do cluster Spark Standalone (Caso B)
- **RustFS S3 API (9000):** storage compatível com S3 usado no Caso D
- **YARN RM (8088):** gerenciador de recursos do Hadoop YARN (Caso C)
- **HDFS HttpFS (14000):** gateway HTTP para o HDFS (Caso C)

💡 **Dica:** Você não precisa de todos os perfis rodando ao mesmo tempo. Ative apenas o necessário para o caso que está estudando com `make up-<caso>`.


## Verificação do Dataset (Camada Bronze)

Os Casos B/C/D usam dados na camada Bronze (`data/bronze/`). A célula abaixo verifica se o dataset já foi gerado e, caso não exista, gera automaticamente uma amostra de escala `small`.

📌 A escala `small` é suficiente para todos os notebooks deste laboratório. Dataset maior só é necessário para testes de desempenho com dados volumosos.

In [ ]:
import subprocess
from pathlib import Path

# Caminho para a camada Bronze do Data Lake (dados gerados)
data_dir = Path("../data/bronze")
# Verifica se o diretório existe e não está vazio
if data_dir.exists() and any(data_dir.iterdir()):
    print(f"✅ Dataset found at {data_dir.resolve()}")
    # Confirma a presença de cada tabela esperada no dataset
    for table in ["empresas", "funcionarios", "vendas"]:
        print(f"   - {table}: {'present' if (data_dir / table).exists() else 'MISSING'}")
else:
    print("⬜ No dataset found. Generating scale=small now...")
    # Gera o dataset automaticamente com escala pequena via script dedicado
    subprocess.run(
        ["uv", "run", "python", "scripts/generate_dataset.py", "--scale", "small"],
        cwd="..",
        check=True,
    )

📌 **Estrutura do dataset (camada Bronze):**
- `empresas/` — cadastro de empresas com setor de atuação
- `funcionarios/` — funcionários com cargo, salário e data de admissão
- `vendas/` — transações com valor, data e referências aos funcionários

Os próximos notebooks lêem esses mesmos arquivos. Se o dataset foi gerado agora, pode prosseguir tranquilamente.

🏁 **Setup completo!** Você já pode abrir e executar o notebook 01.
